# Global 3D t-SNE from the Original Velocity Space

This notebook repeats the global t-SNE visualization without PCA preprocessing. Each snapshot is represented directly by the original fluctuating velocity vector

$$
\mathbf{x}_i = \bigl(u_i(x_1,y_1),\ldots,u_i(x_{n_g},y_{n_g}),v_i(x_1,y_1),\ldots,v_i(x_{n_g},y_{n_g})\bigr) \in \mathbb{R}^{171\,622},
$$

where $n_g=269\times319$. The goal is to test whether the global organization by actuation condition is already present in the Euclidean geometry of the original velocity fields, without PCA, feature masking, spatial downsampling, standardization, or PCA initialization.

For the complete dataset, the computation is very large: all runs contain $50\,751$ snapshots and the original-space snapshot matrix would require about $35$ GB in `float32`, while a full pairwise distance matrix would require about $10$ GB in `float32`. For this reason, the default configuration uses a balanced deterministic subsample from every run. Setting `MAX_SNAPSHOTS_PER_RUN = None` and `ALLOW_FULL_GLOBAL_RUN = True` switches the notebook to the full all-snapshot experiment, but this is expected to be computationally expensive.

The notebook produces two views of the same three-dimensional embedding: one colored by run number, and one colored by actuation phase for the forced runs only. RUN1 is excluded from the phase-colored view because it is unforced.


In [ ]:
from pathlib import Path
from time import time
import gc
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import ScalarMappable
from sklearn.metrics import pairwise_distances
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------------------------------------------------------
# Tunable experiment settings
# -----------------------------------------------------------------------------
RUN_IDS = tuple(range(1, 26))
N_COMPONENTS = 3
N_PHASES = 20

# The default is a balanced global subsample. Set to None for the full dataset.
MAX_SNAPSHOTS_PER_RUN = 80
SAMPLING_MODE = "uniform"          # "uniform" or "first"
ALLOW_FULL_GLOBAL_RUN = False       # Must be True if MAX_SNAPSHOTS_PER_RUN is None.

# t-SNE parameters. The random initialization avoids hidden PCA preprocessing.
PERPLEXITY = 50
N_TRIALS = 1
LEARNING_RATE = 200.0
MAX_ITER = 1000
RANDOM_STATE_START = 0
INIT = "random"
METHOD = "barnes_hut"
ANGLE = 0.5

# Distance handling. Precomputing is useful for moderate subsamples and repeated trials.
# For the full global dataset this would require about 10 GB and very high compute time.
MAX_PRECOMPUTED_DISTANCE_POINTS = 8000

# Plotting parameters.
POINT_SIZE_BY_RUN = 3.0
POINT_SIZE_BY_PHASE = 3.0
POINT_ALPHA_BY_RUN = 0.72
POINT_ALPHA_BY_PHASE = 0.88
FIGSIZE_BY_RUN = (8.6, 7.0)
FIGSIZE_BY_PHASE = (8.6, 7.0)
DPI = 300
RUN_CMAP_NAME = "turbo"
PHASE_CMAP_NAME = "twilight_shifted"
VIEW_ELEV = 24
VIEW_AZIM = -58

# Memory safety. If the selected original-space matrix is larger than this,
# it is stored as an on-disk memmap rather than as a normal in-memory array.
MAX_IN_MEMORY_MATRIX_GB = 8.0


# -----------------------------------------------------------------------------
# File locations
# -----------------------------------------------------------------------------
# The candidate search makes the notebook robust to being launched either from
# the repository root (`X:/TFG_datos`) or from the code folder itself.
for candidate in (Path.cwd(), Path.cwd() / "code"):
    if (candidate / "compressed_data").exists() and (candidate / "data" / "info.txt").exists():
        CODE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "Could not locate the code folder. Run this notebook from the repository "
        "root or from the code folder."
    )

COMPRESSED_DIR = CODE_DIR / "compressed_data"
INFO_PATH = CODE_DIR / "data" / "info.txt"
TSNE_DIR = CODE_DIR / "tsne_data"
FIGURE_DIR = CODE_DIR / "figures" / "tsne_experiments"
TSNE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sample_tag = "all" if MAX_SNAPSHOTS_PER_RUN is None else f"max{MAX_SNAPSHOTS_PER_RUN}"
settings_tag = f"{sample_tag}_perp{PERPLEXITY}_trials{N_TRIALS}"

EMBEDDING_CACHE = TSNE_DIR / f"global_tsne_original_space_3D_{settings_tag}.npz"
METADATA_PATH = TSNE_DIR / f"global_tsne_original_space_3D_{settings_tag}.json"
DISTANCE_CACHE = TSNE_DIR / f"global_original_space_distances_{settings_tag}_float32.npz"
MEMMAP_PATH = TSNE_DIR / f"global_original_space_matrix_{sample_tag}_float32.dat"

OUTPUT_STEM_RUN = f"global_tsne_original_space_3d_by_run_{settings_tag}"
OUTPUT_STEM_PHASE = f"global_tsne_original_space_3d_forced_phase_{settings_tag}"


def parse_run_info(info_path: Path) -> dict[int, float]:
    """Read the actuation Strouhal number associated with each run."""
    run_to_st = {}
    pattern = re.compile(r"\*?Run\s+(\d+)\s+([0-9.]+)")
    for line in info_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        match = pattern.search(line)
        if match:
            run_to_st[int(match.group(1))] = float(match.group(2))
    return run_to_st


def expected_snapshot_count(run_id: int) -> int:
    """Return the known compressed-data snapshot count for each run."""
    # RUN1 currently stores 2031 snapshots; RUN2-RUN25 store 2030.
    return 2031 if run_id == 1 else 2030


def select_snapshot_indices(n_snapshots: int, max_snapshots: int | None, mode: str) -> np.ndarray:
    """Select a deterministic subset of snapshots for one run."""
    if max_snapshots is None or max_snapshots >= n_snapshots:
        return np.arange(n_snapshots, dtype=np.int32)

    if mode == "uniform":
        # Uniform sampling preserves the full temporal extent of the run.
        indices = np.linspace(0, n_snapshots - 1, max_snapshots, dtype=np.int32)
        return np.unique(indices)

    if mode == "first":
        return np.arange(max_snapshots, dtype=np.int32)

    raise ValueError(f"Unknown SAMPLING_MODE: {mode}")


def make_sample_plan(run_ids: tuple[int, ...]) -> pd.DataFrame:
    """Build the run-wise sampling plan used for the global embedding."""
    rows = []
    for run_id in run_ids:
        n_total = expected_snapshot_count(run_id)
        indices = select_snapshot_indices(n_total, MAX_SNAPSHOTS_PER_RUN, SAMPLING_MODE)
        rows.append({
            "run_id": run_id,
            "n_total": n_total,
            "n_selected": len(indices),
            "selected_indices": indices,
        })
    return pd.DataFrame(rows)


def phase_labels_for_indices(run_id: int, indices: np.ndarray) -> np.ndarray:
    """Return phase labels for selected snapshots; RUN1 receives -1."""
    if run_id == 1:
        return np.full(indices.shape, -1, dtype=np.int16)
    return (indices % N_PHASES).astype(np.int16)


def estimate_memory_gb(n_snapshots: int, ambient_dim: int) -> dict[str, float]:
    """Estimate memory requirements for the selected original-space experiment."""
    return {
        "snapshot_matrix_float32_gb": n_snapshots * ambient_dim * 4 / 1e9,
        "distance_matrix_float32_gb": n_snapshots * n_snapshots * 4 / 1e9,
        "distance_matrix_float64_gb": n_snapshots * n_snapshots * 8 / 1e9,
    }


def load_global_original_matrix(sample_plan: pd.DataFrame):
    """Load selected original-space snapshots from all requested runs.

    Each selected snapshot is represented as the concatenation of all `u` coordinates
    followed by all `v` coordinates. The output matrix contains original velocity
    coordinates only: there is no PCA, feature masking, spatial downsampling or
    standardization.
    """
    run_to_st = parse_run_info(INFO_PATH)

    # The grid shape is fixed by the experiment.
    n_y, n_x = 269, 319
    n_grid = n_y * n_x
    ambient_dim = 2 * n_grid
    n_selected_total = int(sample_plan["n_selected"].sum())
    memory = estimate_memory_gb(n_selected_total, ambient_dim)

    print("Selected global original-space experiment")
    print(f"  selected snapshots: {n_selected_total}")
    print(f"  ambient dimension: {ambient_dim}")
    print(f"  snapshot matrix estimate: {memory['snapshot_matrix_float32_gb']:.2f} GB")
    print(f"  full distance matrix estimate: {memory['distance_matrix_float32_gb']:.2f} GB")

    if MAX_SNAPSHOTS_PER_RUN is None and not ALLOW_FULL_GLOBAL_RUN:
        raise RuntimeError(
            "The full all-snapshot original-space global t-SNE was requested, but "
            "ALLOW_FULL_GLOBAL_RUN is False. This protects against accidentally "
            "starting a very expensive computation. Set ALLOW_FULL_GLOBAL_RUN = True "
            "only if you intentionally want to run the full experiment."
        )

    use_memmap = memory["snapshot_matrix_float32_gb"] > MAX_IN_MEMORY_MATRIX_GB
    if use_memmap:
        print(f"  using on-disk memmap: {MEMMAP_PATH}")
        X = np.memmap(MEMMAP_PATH, dtype=np.float32, mode="w+", shape=(n_selected_total, ambient_dim))
    else:
        X = np.empty((n_selected_total, ambient_dim), dtype=np.float32)

    run_labels = np.empty(n_selected_total, dtype=np.int16)
    st_labels = np.empty(n_selected_total, dtype=np.float32)
    phase_labels = np.empty(n_selected_total, dtype=np.int16)
    source_indices = np.empty(n_selected_total, dtype=np.int32)

    offset = 0
    t0 = time()
    for row in sample_plan.itertuples(index=False):
        run_id = int(row.run_id)
        indices = np.asarray(row.selected_indices, dtype=np.int32)
        n_selected = len(indices)
        run_path = COMPRESSED_DIR / f"RUN{run_id}_PIV_compressed.npz"

        print(f"Loading RUN{run_id}: {n_selected}/{row.n_total} snapshots")
        if not run_path.exists():
            raise FileNotFoundError(f"Missing compressed data file: {run_path}")

        with np.load(run_path) as data:
            # Compressed NPZ files are loaded one velocity component at a time.
            # This keeps peak memory lower than loading both components together.
            u_all = data["u"]
            u_selected = np.asarray(u_all[indices], dtype=np.float32)
            X[offset:offset + n_selected, :n_grid] = u_selected.reshape(n_selected, n_grid)
            del u_all, u_selected
            gc.collect()

            v_all = data["v"]
            v_selected = np.asarray(v_all[indices], dtype=np.float32)
            X[offset:offset + n_selected, n_grid:] = v_selected.reshape(n_selected, n_grid)
            del v_all, v_selected
            gc.collect()

        run_labels[offset:offset + n_selected] = run_id
        st_labels[offset:offset + n_selected] = run_to_st[run_id]
        phase_labels[offset:offset + n_selected] = phase_labels_for_indices(run_id, indices)
        source_indices[offset:offset + n_selected] = indices
        offset += n_selected

    if use_memmap:
        X.flush()

    metadata = {
        "run_ids": list(map(int, RUN_IDS)),
        "max_snapshots_per_run": MAX_SNAPSHOTS_PER_RUN,
        "sampling_mode": SAMPLING_MODE,
        "n_snapshots_selected": n_selected_total,
        "ambient_dimension": ambient_dim,
        "grid_shape": [n_y, n_x],
        "memory_estimates_gb": memory,
        "use_memmap": bool(use_memmap),
        "memmap_path": str(MEMMAP_PATH) if use_memmap else None,
        "load_time_sec": float(time() - t0),
    }
    return X, run_labels, st_labels, phase_labels, source_indices, metadata


def compute_or_load_distance_matrix(X: np.ndarray | np.memmap) -> tuple[np.ndarray | None, dict]:
    """Compute or load original-space Euclidean distances when feasible."""
    n_samples = X.shape[0]
    if n_samples > MAX_PRECOMPUTED_DISTANCE_POINTS:
        return None, {
            "distance_mode": "direct_metric_euclidean",
            "distance_cache": None,
            "distance_time_sec": None,
        }

    if DISTANCE_CACHE.exists():
        with np.load(DISTANCE_CACHE) as data:
            distances = data["distances"].astype(np.float32, copy=False)
            distance_metadata = json.loads(str(data["metadata"].item()))
        distance_metadata["loaded_from_distance_cache"] = True
        return distances, distance_metadata

    t0 = time()
    print("Computing original-space Euclidean distance matrix...")
    distances = pairwise_distances(X, metric="euclidean", n_jobs=-1).astype(np.float32, copy=False)
    np.fill_diagonal(distances, 0.0)

    distance_metadata = {
        "distance_mode": "precomputed_euclidean",
        "distance_cache": str(DISTANCE_CACHE),
        "distance_time_sec": float(time() - t0),
        "loaded_from_distance_cache": False,
    }
    np.savez_compressed(
        DISTANCE_CACHE,
        distances=distances,
        metadata=json.dumps(distance_metadata),
    )
    return distances, distance_metadata


def compute_or_load_tsne_embedding() -> tuple[dict, dict]:
    """Compute or load the global original-space 3D t-SNE embedding."""
    if EMBEDDING_CACHE.exists():
        with np.load(EMBEDDING_CACHE) as data:
            result = {
                "embedding": data["embedding"].astype(np.float32),
                "run_labels": data["run_labels"].astype(np.int16),
                "st_labels": data["st_labels"].astype(np.float32),
                "phase_labels": data["phase_labels"].astype(np.int16),
                "source_indices": data["source_indices"].astype(np.int32),
                "best_kl": float(data["best_kl"]),
                "best_trial": int(data["best_trial"]),
                "all_kls": data["all_kls"].astype(float),
                "loaded_from_embedding_cache": True,
            }
            metadata = json.loads(str(data["metadata"].item()))
        return result, metadata

    sample_plan = make_sample_plan(RUN_IDS)
    X, run_labels, st_labels, phase_labels, source_indices, metadata = load_global_original_matrix(sample_plan)
    distances, distance_metadata = compute_or_load_distance_matrix(X)
    metadata.update(distance_metadata)

    tsne_input = distances if distances is not None else X
    metric = "precomputed" if distances is not None else "euclidean"

    embeddings = []
    kl_values = []
    t0 = time()

    for trial in range(N_TRIALS):
        random_state = RANDOM_STATE_START + trial
        print(f"Running global original-space 3D t-SNE trial {trial + 1}/{N_TRIALS} "
              f"with random_state={random_state}...")

        model = TSNE(
            n_components=N_COMPONENTS,
            perplexity=PERPLEXITY,
            learning_rate=LEARNING_RATE,
            max_iter=MAX_ITER,
            init=INIT,
            metric=metric,
            method=METHOD,
            angle=ANGLE,
            random_state=random_state,
            verbose=1,
        )
        embedding = model.fit_transform(tsne_input).astype(np.float32)
        embeddings.append(embedding)
        kl_values.append(float(model.kl_divergence_))
        print(f"  KL divergence: {model.kl_divergence_:.4f}")

    all_kls = np.asarray(kl_values, dtype=float)
    best_trial = int(np.argmin(all_kls))
    best_embedding = embeddings[best_trial]

    metadata.update({
        "n_components": N_COMPONENTS,
        "perplexity": PERPLEXITY,
        "n_trials": N_TRIALS,
        "learning_rate": LEARNING_RATE,
        "max_iter": MAX_ITER,
        "init": INIT,
        "method": METHOD,
        "angle": ANGLE,
        "random_state_start": RANDOM_STATE_START,
        "tsne_metric": metric,
        "tsne_time_sec": float(time() - t0),
        "best_trial": best_trial,
        "best_kl": float(all_kls[best_trial]),
        "all_kls": all_kls.tolist(),
        "loaded_from_embedding_cache": False,
    })

    np.savez_compressed(
        EMBEDDING_CACHE,
        embedding=best_embedding,
        run_labels=run_labels,
        st_labels=st_labels,
        phase_labels=phase_labels,
        source_indices=source_indices,
        best_kl=float(all_kls[best_trial]),
        best_trial=best_trial,
        all_kls=all_kls,
        metadata=json.dumps(metadata),
    )
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    result = {
        "embedding": best_embedding,
        "run_labels": run_labels,
        "st_labels": st_labels,
        "phase_labels": phase_labels,
        "source_indices": source_indices,
        "best_kl": float(all_kls[best_trial]),
        "best_trial": best_trial,
        "all_kls": all_kls,
        "loaded_from_embedding_cache": False,
    }
    return result, metadata


def set_3d_axes_equal_to_data(ax, points, robust_percentiles=(1, 99), pad_fraction=0.06):
    """Set comparable x, y, z limits using robust percentiles of the point cloud."""
    low = np.percentile(points, robust_percentiles[0], axis=0)
    high = np.percentile(points, robust_percentiles[1], axis=0)
    center = 0.5 * (low + high)
    radius = 0.5 * np.max(high - low) * (1.0 + pad_fraction)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def make_run_cmap(run_ids):
    """Return a categorical colormap for integer run labels."""
    run_ids = np.asarray(tuple(run_ids), dtype=int)
    colors = plt.get_cmap(RUN_CMAP_NAME)(np.linspace(0.04, 0.96, len(run_ids)))
    cmap = ListedColormap(colors, name="run_labels")
    boundaries = np.r_[run_ids - 0.5, run_ids[-1] + 0.5]
    norm = BoundaryNorm(boundaries, cmap.N)
    return cmap, norm


def make_phase_cmap(n_phases: int = 20):
    """Return a cyclic discrete colormap for phase labels."""
    colors = plt.get_cmap(PHASE_CMAP_NAME)(np.linspace(0.0, 1.0, n_phases, endpoint=False))
    cmap = ListedColormap(colors, name=f"phase_{n_phases}")
    norm = BoundaryNorm(np.arange(-0.5, n_phases + 0.5, 1.0), cmap.N)
    return cmap, norm


def plot_embedding_by_run(result: dict):
    """Plot the global 3D embedding with one color per run."""
    embedding = result["embedding"]
    run_labels = result["run_labels"]
    run_cmap, run_norm = make_run_cmap(RUN_IDS)

    fig = plt.figure(figsize=FIGSIZE_BY_RUN, constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(
        embedding[:, 0], embedding[:, 1], embedding[:, 2],
        c=run_labels,
        cmap=run_cmap,
        norm=run_norm,
        s=POINT_SIZE_BY_RUN,
        alpha=POINT_ALPHA_BY_RUN,
        linewidths=0,
        depthshade=False,
        rasterized=True,
    )

    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
    set_3d_axes_equal_to_data(ax, embedding)
    ax.set_xlabel("t-SNE 1", labelpad=8, fontsize=12)
    ax.set_ylabel("t-SNE 2", labelpad=8, fontsize=12)
    ax.set_zlabel("t-SNE 3", labelpad=8, fontsize=12)
    ax.set_title("Global original-space 3D t-SNE colored by run", fontsize=14)
    ax.tick_params(labelsize=10)

    cbar = fig.colorbar(scatter, ax=ax, pad=0.04, fraction=0.045)
    cbar.set_label("Run", fontsize=12)
    cbar.set_ticks([1, 5, 10, 15, 20, 25])
    cbar.ax.tick_params(labelsize=10)

    png_path = FIGURE_DIR / f"{OUTPUT_STEM_RUN}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM_RUN}.pdf"
    fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    return png_path, pdf_path


def plot_forced_embedding_by_phase(result: dict):
    """Plot the forced-run points from the same embedding colored by phase."""
    embedding = result["embedding"]
    phase_labels = result["phase_labels"]
    forced_mask = phase_labels >= 0
    phase_cmap, phase_norm = make_phase_cmap(N_PHASES)

    fig = plt.figure(figsize=FIGSIZE_BY_PHASE, constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(
        embedding[forced_mask, 0], embedding[forced_mask, 1], embedding[forced_mask, 2],
        c=phase_labels[forced_mask],
        cmap=phase_cmap,
        norm=phase_norm,
        s=POINT_SIZE_BY_PHASE,
        alpha=POINT_ALPHA_BY_PHASE,
        linewidths=0,
        depthshade=False,
        rasterized=True,
    )

    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
    set_3d_axes_equal_to_data(ax, embedding[forced_mask])
    ax.set_xlabel("t-SNE 1", labelpad=8, fontsize=12)
    ax.set_ylabel("t-SNE 2", labelpad=8, fontsize=12)
    ax.set_zlabel("t-SNE 3", labelpad=8, fontsize=12)
    ax.set_title("Global original-space 3D t-SNE colored by phase", fontsize=14)
    ax.tick_params(labelsize=10)

    cbar = fig.colorbar(scatter, ax=ax, pad=0.04, fraction=0.045)
    cbar.set_label("Actuation phase", fontsize=12)
    cbar.set_ticks([0, 5, 10, 15, 19])
    cbar.ax.tick_params(labelsize=10)

    png_path = FIGURE_DIR / f"{OUTPUT_STEM_PHASE}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM_PHASE}.pdf"
    fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    return png_path, pdf_path


result, metadata = compute_or_load_tsne_embedding()
run_png_path, run_pdf_path = plot_embedding_by_run(result)
phase_png_path, phase_pdf_path = plot_forced_embedding_by_phase(result)

print("Global original-space 3D t-SNE")
print(f"  embedding cache: {EMBEDDING_CACHE}")
print(f"  loaded from embedding cache: {result['loaded_from_embedding_cache']}")
print(f"  selected snapshots: {metadata['n_snapshots_selected']}")
print(f"  ambient dimension: {metadata['ambient_dimension']}")
print(f"  best trial: {result['best_trial'] + 1}/{N_TRIALS}")
print(f"  best KL divergence: {result['best_kl']:.4f}")
print(f"  all KL values: {np.array2string(result['all_kls'], precision=4)}")
print(f"  saved run-colored PNG: {run_png_path}")
print(f"  saved run-colored PDF: {run_pdf_path}")
print(f"  saved phase-colored PNG: {phase_png_path}")
print(f"  saved phase-colored PDF: {phase_pdf_path}")


## Interpretation

Run the implementation cell above before writing the interpretation. The interpretation should report the number of selected snapshots, the final KL divergence, whether snapshots from the same run occupy coherent regions in the embedding, and whether the forced-run phase coloring shows visible cyclic organization. If the default balanced subsample is used, the text must explicitly state that the figure is an exploratory global subsample, not the full all-snapshot embedding.
